In [1]:
!whoami
!echo 24BAD405 EX10d

nishanth\nishanth
24BAD405 EX10d


In [1]:
import heapq

INF = float('inf')


# -------------------------------
# Task 2: Node Structure
# -------------------------------
class Node:
    def __init__(self, matrix, path, level, vertex, cost):
        self.matrix = matrix
        self.path = path
        self.level = level
        self.vertex = vertex
        self.cost = cost   # lower bound

    def __lt__(self, other):
        return self.cost < other.cost


# -------------------------------
# Task 1: Matrix Reduction
# -------------------------------
def reduce_matrix(matrix):
    n = len(matrix)
    reduction_cost = 0

    # Row reduction
    for i in range(n):
        row_min = min(matrix[i])
        if row_min != INF and row_min > 0:
            reduction_cost += row_min
            for j in range(n):
                if matrix[i][j] != INF:
                    matrix[i][j] -= row_min

    # Column reduction
    for j in range(n):
        col_min = min(matrix[i][j] for i in range(n))
        if col_min != INF and col_min > 0:
            reduction_cost += col_min
            for i in range(n):
                if matrix[i][j] != INF:
                    matrix[i][j] -= col_min

    return reduction_cost


# -------------------------------
# Create Child Node
# -------------------------------
def create_child(parent, original_cost, from_city, to_city):
    n = len(parent.matrix)
    new_matrix = [row[:] for row in parent.matrix]

    # Block row and column
    for j in range(n):
        new_matrix[from_city][j] = INF
    for i in range(n):
        new_matrix[i][to_city] = INF

    # Prevent returning to start too early
    new_matrix[to_city][0] = INF

    # Calculate new lower bound
    edge_cost = original_cost[from_city][to_city]
    reduction_cost = reduce_matrix(new_matrix)

    child_cost = parent.cost + edge_cost + reduction_cost

    return Node(
        new_matrix,
        parent.path + [to_city],
        parent.level + 1,
        to_city,
        child_cost
    )


# -------------------------------
# Task 3: Branch and Bound TSP
# -------------------------------
def solve_tsp(cost_matrix):
    n = len(cost_matrix)

    root_matrix = [row[:] for row in cost_matrix]
    root_cost = reduce_matrix(root_matrix)

    root = Node(root_matrix, [0], 0, 0, root_cost)

    pq = []
    heapq.heappush(pq, root)

    best_cost = INF
    best_path = []

    while pq:
        current = heapq.heappop(pq)

        if current.cost >= best_cost:
            continue

        # If all cities visited
        if current.level == n - 1:
            last = current.vertex
            if cost_matrix[last][0] != INF:
                final_cost = current.cost + cost_matrix[last][0]
                if final_cost < best_cost:
                    best_cost = final_cost
                    best_path = current.path + [0]
            continue

        # Expand children
        for next_city in range(n):
            if next_city not in current.path and cost_matrix[current.vertex][next_city] != INF:
                child = create_child(current, cost_matrix, current.vertex, next_city)
                if child.cost < best_cost:
                    heapq.heappush(pq, child)

    return best_cost, best_path


# -------------------------------
# MAIN
# -------------------------------
graph1 = [
    [INF, 10, 15, 20],
    [10, INF, 35, 25],
    [15, 35, INF, 30],
    [20, 25, 30, INF]
]

cost1, path1 = solve_tsp(graph1)
print("Graph 1")
print("Minimum Cost:", int(cost1))
print("Optimal Path:", " -> ".join(map(str, path1)))

print()

graph2 = [
    [INF, 20, 30, 10, 11],
    [15, INF, 16, 4, 2],
    [3, 5, INF, 2, 4],
    [19, 6, 18, INF, 3],
    [16, 4, 7, 16, INF]
]

cost2, path2 = solve_tsp(graph2)
print("Graph 2")
print("Minimum Cost:", int(cost2))
print("Optimal Path:", " -> ".join(map(str, path2)))

Graph 1
Minimum Cost: 160
Optimal Path: 0 -> 1 -> 3 -> 2 -> 0

Graph 2
Minimum Cost: 53
Optimal Path: 0 -> 3 -> 1 -> 4 -> 2 -> 0
